In [1]:
import ast
import pandas as pd

# Current dataset — already known
df_current = pd.read_csv("../data/raw/resume_data_for_ranking.csv", nrows=3)
print("=== resume_data_for_ranking.csv ===")
print(df_current.columns.tolist())
print()

# job_descriptions.csv — large file; load only 5 rows for now
df_jobs_sample = pd.read_csv("../data/raw/job_descriptions.csv", nrows=5)
print("=== job_descriptions.csv ===")
print(df_jobs_sample.columns.tolist())
print()

# Resume.csv
df_resumes_sample = pd.read_csv("../data/raw/Resume.csv", nrows=5)
print("=== Resume.csv ===")
print(df_resumes_sample.columns.tolist())

=== resume_data_for_ranking.csv ===
['address', 'career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'company_urls', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'extra_curricular_organization_links', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'online_links', 'issue_dates', 'expiry_dates', 'job_position_name', 'educationaL_requirements', 'experiencere_requirement', 'age_requirement', 'responsibilities.1', 'skills_required', 'matched_score']

=== job_descriptions.csv ===
['Job Id', 'Experience', 'Qualifications', 'Salary Range', 'location', 'Country', 'latitude', 'longitude', 'Work Type', 'Company Size', 'Job Posting Date', 'Preference', 'Contact Person', 'Contact', '

In [2]:
print("--- job_descriptions.csv: skills/Responsibilities/Qualifications preview ---")
for col in ["skills", "Responsibilities", "Qualifications", "Job Title", "Experience"]:
    if col in df_jobs_sample.columns:
        print(f"{col}: {df_jobs_sample[col].iloc[0]}")
print()

print("--- Resume.csv: Resume_str preview ---")
print(df_resumes_sample["Resume_str"].iloc[0][:500])

--- job_descriptions.csv: skills/Responsibilities/Qualifications preview ---
skills: Social media platforms (e.g., Facebook, Twitter, Instagram) Content creation and scheduling Social media analytics and insights Community engagement Paid social advertising
Responsibilities: Manage and grow social media accounts, create engaging content, and interact with the online community. Develop social media content calendars and strategies. Monitor social media trends and engagement metrics.
Qualifications: M.Tech
Job Title: Digital Marketing Specialist
Experience: 5 to 15 Years

--- Resume.csv: Resume_str preview ---
         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfactio

In [3]:
import re
import numpy as np

# --- Canonical schema ---
CANONICAL_RESUME_FIELDS = [
    "candidate_summary", "skills_list", "education_text",
    "candidate_experience_years", "raw_full_text", "source_dataset",
]
CANONICAL_JOB_FIELDS = [
    "job_title", "responsibilities_text", "required_skills_list",
    "education_requirements_text", "experience_required_years",
    "raw_full_text", "source_dataset",
]


def normalize_skill(skill):
    return " ".join(str(skill).strip().lower().split())


def safe_literal_list(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []


# --- Mapping 1: resume_data_for_ranking.csv (resume) ---

def map_current_resume_to_canonical(row):
    skills = [normalize_skill(s) for s in safe_literal_list(row.get("skills")) if str(s).strip()]
    parts = [str(row.get("career_objective", "")), " ".join(skills)]
    return {
        "candidate_summary": row.get("career_objective") if pd.notna(row.get("career_objective")) else None,
        "skills_list": skills if skills else None,
        "education_text": row.get("educational_results") if pd.notna(row.get("educational_results")) else None,
        "candidate_experience_years": None,
        "raw_full_text": " ".join(p for p in parts if p).strip(),
        "source_dataset": "resume_data_for_ranking",
    }


# --- Mapping 1bis: resume_data_for_ranking.csv (job) ---

def map_current_job_to_canonical(row):
    skills_required = [normalize_skill(s) for s in str(row.get("skills_required", "")).split("\n") if s.strip()] \
        if pd.notna(row.get("skills_required")) else []
    exp_match = re.search(r"(\d+)", str(row.get("experiencere_requirement", "")))
    parts = [str(row.get("responsibilities.1", "")), " ".join(skills_required)]
    return {
        "job_title": row.get("job_position_name") if pd.notna(row.get("job_position_name")) else None,
        "responsibilities_text": row.get("responsibilities.1") if pd.notna(row.get("responsibilities.1")) else None,
        "required_skills_list": skills_required if skills_required else None,
        "education_requirements_text": row.get("educationaL_requirements") if pd.notna(row.get("educationaL_requirements")) else None,
        "experience_required_years": float(exp_match.group(1)) if exp_match else None,
        "raw_full_text": " ".join(p for p in parts if p).strip(),
        "source_dataset": "resume_data_for_ranking",
    }


# --- Mapping 2: job_descriptions.csv (job postings only) ---

def map_job_descriptions_to_canonical(row):
    exp_match = re.search(r"(\d+)", str(row.get("Experience", "")))
    parts = [
        str(row.get("Responsibilities", "")),
        str(row.get("skills", "")),
        str(row.get("Qualifications", "")),
    ]
    return {
        "job_title": row.get("Job Title") if pd.notna(row.get("Job Title")) else None,
        "responsibilities_text": row.get("Responsibilities") if pd.notna(row.get("Responsibilities")) else None,
        "required_skills_list": None,
        "education_requirements_text": row.get("Qualifications") if pd.notna(row.get("Qualifications")) else None,
        "experience_required_years": float(exp_match.group(1)) if exp_match else None,
        "raw_full_text": " ".join(p for p in parts if p and p != "nan").strip(),
        "source_dataset": "job_descriptions",
    }


# --- Mapping 3: Resume.csv (resumes only, raw text) ---

def map_resume_csv_to_canonical(row):
    text = row.get("Resume_str") if pd.notna(row.get("Resume_str")) else ""
    return {
        "candidate_summary": None,
        "skills_list": None,
        "education_text": None,
        "candidate_experience_years": None,
        "raw_full_text": text.strip() if text else "",
        "source_dataset": "resume_csv",
    }

In [4]:
print("=== Canonical resume (resume_data_for_ranking) ===")
print(map_current_resume_to_canonical(df_current.iloc[0]))
print()

print("=== Canonical job (resume_data_for_ranking) ===")
print(map_current_job_to_canonical(df_current.iloc[0]))
print()

print("=== Canonical job (job_descriptions) ===")
print(map_job_descriptions_to_canonical(df_jobs_sample.iloc[0]))
print()

print("=== Canonical resume (Resume.csv) ===")
result = map_resume_csv_to_canonical(df_resumes_sample.iloc[0])
result["raw_full_text"] = result["raw_full_text"][:200] + "..."
print(result)

=== Canonical resume (resume_data_for_ranking) ===
{'candidate_summary': 'Big data analytics working and database warehouse manager with robust experience in handling all kinds of data. I have also used multiple cloud infrastructure services and am well acquainted with them. Currently in search of role that offers more of development.', 'skills_list': ['big data', 'hadoop', 'hive', 'python', 'mapreduce', 'spark', 'java', 'machine learning', 'cloud', 'hdfs', 'yarn', 'core java', 'data science', 'c++', 'data structures', 'dbms', 'rdbms', 'informatica', 'talend', 'amazon redshift', 'microsoft azure'], 'education_text': "['N/A']", 'candidate_experience_years': None, 'raw_full_text': 'Big data analytics working and database warehouse manager with robust experience in handling all kinds of data. I have also used multiple cloud infrastructure services and am well acquainted with them. Currently in search of role that offers more of development. big data hadoop hive python mapreduce spark java

In [5]:
df_current_full = pd.read_csv("../data/raw/resume_data_for_ranking.csv")
df_jobs_full = pd.read_csv("../data/raw/job_descriptions.csv", nrows=50_000)
df_resumes_full = pd.read_csv("../data/raw/Resume.csv")

canonical_resumes_current = df_current_full.apply(map_current_resume_to_canonical, axis=1, result_type="expand")
canonical_jobs_current = df_current_full.apply(map_current_job_to_canonical, axis=1, result_type="expand")
canonical_jobs_external = df_jobs_full.apply(map_job_descriptions_to_canonical, axis=1, result_type="expand")
canonical_resumes_external = df_resumes_full.apply(map_resume_csv_to_canonical, axis=1, result_type="expand")

print("CV (actuel)      :", canonical_resumes_current.shape)
print("Current jobs  :", canonical_jobs_current.shape)
print("External jobs :", canonical_jobs_external.shape)
print("External resumes     :", canonical_resumes_external.shape)

CV (actuel)      : (9544, 6)
Current jobs  : (9544, 7)
External jobs : (50000, 7)
External resumes     : (2484, 6)


In [6]:
for name, cdf in [
    ("Current resumes", canonical_resumes_current),
    ("Current jobs", canonical_jobs_current),
    ("External jobs", canonical_jobs_external),
    ("External resumes", canonical_resumes_external),
]:
    empty_rate = (cdf["raw_full_text"].str.strip() == "").mean()
    print(f"{name} : {empty_rate:.1%} de raw_full_text vide")

Current resumes : 0.0% of raw_full_text values that are empty
Current jobs : 0.0% of raw_full_text values that are empty
External jobs : 0.0% of raw_full_text values that are empty
External resumes : 0.0% of raw_full_text values that are empty


In [7]:
canonical_resumes_current.to_parquet("../data/processed/canonical_resumes_current.parquet")
canonical_jobs_current.to_parquet("../data/processed/canonical_jobs_current.parquet")
canonical_jobs_external.to_parquet("../data/processed/canonical_jobs_external.parquet")
canonical_resumes_external.to_parquet("../data/processed/canonical_resumes_external.parquet")

print("Canonical tables saved.")

Canonical tables saved.


In [8]:
role_counts = pd.Series(dtype=int)

for chunk in pd.read_csv("../data/raw/job_descriptions.csv", usecols=["Role"], chunksize=200_000):
    role_counts = role_counts.add(chunk["Role"].value_counts(), fill_value=0)

role_counts = role_counts.sort_values(ascending=False)
print(f"Number of distinct roles : {len(role_counts)}")
print(role_counts.head(30))

Number of distinct roles : 376
Role
Interaction Designer             20580.0
Network Administrator            17470.0
User Interface Designer          14036.0
Social Media Manager             13945.0
User Experience Designer         13935.0
Procurement Analyst              13757.0
Social Media Analyst             10659.0
Quality Assurance Analyst        10541.0
SEO Specialist                   10512.0
Executive Assistant              10496.0
Database Administrator           10482.0
Procurement Manager              10407.0
Data Analyst                     10406.0
Backend Developer                10404.0
Demand Planner                   10362.0
Office Manager                   10361.0
Frontend Developer               10308.0
Customer Success Manager         10308.0
Retirement Planner               10305.0
Account Executive                 7063.0
Inside Sales Representative       7052.0
UX/UI Designer                    7028.0
Network Security Analyst          7027.0
Paralegal            

In [9]:
MAX_PER_ROLE = 300  # adjustable based on the number of distinct roles found

sampled_chunks = []
role_seen_count = {}

for chunk in pd.read_csv("../data/raw/job_descriptions.csv", chunksize=200_000):
    for role, group in chunk.groupby("Role"):
        already = role_seen_count.get(role, 0)
        remaining = MAX_PER_ROLE - already
        if remaining <= 0:
            continue
        take = group.sample(n=min(remaining, len(group)), random_state=42)
        sampled_chunks.append(take)
        role_seen_count[role] = already + len(take)

df_jobs_full = pd.concat(sampled_chunks, ignore_index=True)
print(f"Final sample : {df_jobs_full.shape}")
print(f"Number of represented roles : {df_jobs_full['Role'].nunique()}")
print(df_jobs_full["Role"].value_counts().head(20))

Final sample : (112800, 23)
Number of represented roles : 376
Role
API Developer                    300
Accessibility Developer          300
Account Executive                300
Account Manager                  300
Account Strategist               300
Accounting Controller            300
Accounting Manager               300
Acute Care Nurse Practitioner    300
Addiction Counselor              300
Administrative Assistant         300
Administrative Coordinator       300
Administrative Manager           300
Adult Speech Therapist           300
Advertising Account Executive    300
Agile Product Owner              300
Agile Project Manager            300
Aircraft Design Engineer         300
Analytical Chemist               300
Architectural Designer           300
Architectural Drafter            300
Name: count, dtype: int64


In [10]:
canonical_jobs_external = df_jobs_full.apply(map_job_descriptions_to_canonical, axis=1, result_type="expand")

empty_rate = (canonical_jobs_external["raw_full_text"].str.strip() == "").mean()
print(f"% of raw_full_text values that are empty : {empty_rate:.1%}")

canonical_jobs_external.to_parquet("../data/processed/canonical_jobs_external.parquet")
print("Canonical table (diverse external jobs) saved again.")

% of raw_full_text values that are empty : 0.0%
Canonical table (diverse external jobs) saved again.
